# 8장 텍스트 생성: 확률에서 문장을 뽑는 법 (실습)

교재 `docs/book/08-generation.md` 와 함께 본다. 7장에서 학습한 `small-cpu` 의 `best.pt` 를 쓴다. 이 노트북에서 하는 것:

1. 한 스텝의 확률 분포를 눈으로 본다. 모델이 다음에 무엇을 고민하는가
2. temperature · top-k · top-p · 반복 억제가 분포를 어떻게 바꾸는지
3. 같은 프롬프트로 설정을 바꿔 가며 생성, **v1.0 성공 기준**: "옛날 옛적에" 를 말이 되게 잇기

> 전체 실행 약 2분.

In [ ]:
import torch
import torch.nn.functional as F

from shllm.config import TOKENIZER_DIR, setup_cpu
from shllm.generate import adjust_logits, generate, generate_text, next_token_distribution
from shllm.tokenizer import BPETokenizer
from shllm.train import load_checkpoint

setup_cpu()
tok = BPETokenizer.load(TOKENIZER_DIR / "bpe-8192.json")
try:
    model, ck = load_checkpoint("small-cpu", "best.pt")
    print(f"small-cpu best.pt: step {ck['step']}, {model.n_params():,} 파라미터")
except FileNotFoundError:
    model, ck = load_checkpoint("tiny-notebook", "best.pt")
    print("small-cpu 체크포인트가 없어 7장 노트북의 tiny-notebook 을 씁니다 (품질 낮음)")

## 1. 다음 토큰 분포, 모델의 고민

In [ ]:
prompt = "옛날 옛적에 호랑이가"
idx = torch.tensor([tok.encode(prompt)])
probs = next_token_distribution(model, idx)[0]  # (V,)
top = torch.topk(probs, 12)
print(f"{prompt!r} 다음 토큰 상위 12:")
for p, i in zip(top.values, top.indices):
    print(f"  {tok.token_str(int(i))!r:<12} {p.item():6.1%}")
print(f"상위 12 합 {top.values.sum().item():.1%}, 확률 0.1% 이상인 토큰 {(probs > 1e-3).sum().item()}개 / {len(probs):,}")

분포가 놀랄 만큼 **평평하다**, 상위 12개를 다 합쳐도 15% 남짓이고, 0.1% 이상인 토큰이 200개 가까이 된다. 소형 모델이 자신 없는 자리에서 흔히 보이는 모양이다. 매 스텝 이 긴 꼬리에서 하나라도 뽑히면 문장이 산으로 간다. 생성 기법은 대부분 **이 꼬리를 어떻게 다룰 것인가**다.

## 2. 조정 기법 네 가지

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager

cjk = [f.name for f in font_manager.fontManager.ttflist if "CJK" in f.name]
if cjk:
    matplotlib.rcParams["font.family"] = cjk[0]

with torch.no_grad():
    logits, _ = model(idx)
logits = logits[:, -1, :]
settings = {
    "원본 (T=1)": dict(),
    "temperature 0.5": dict(temperature=0.5),
    "temperature 1.5": dict(temperature=1.5),
    "top-k 10": dict(top_k=10),
    "top-p 0.9": dict(top_p=0.9),
}
order = torch.argsort(probs, descending=True)[:30]
fig, ax = plt.subplots(figsize=(10, 4))
for name, kw in settings.items():
    p = F.softmax(adjust_logits(logits, idx, **kw), -1)[0]
    ax.plot(p[order].numpy(), marker=".", label=f"{name}  (0 아닌 토큰 {(p > 0).sum().item():,})")
ax.set_xticks(range(30), [tok.token_str(int(i)).replace(" ", "␣") for i in order], rotation=90, fontsize=8)
ax.set_ylabel("확률")
ax.set_title("같은 logits, 다른 조정, 상위 30 토큰")
ax.legend(fontsize=8)
plt.show()

**그림에서 볼 것**: 원본(T=1) 곡선은 완만하다. T=0.5 는 왼쪽 몇 개가 솟고 꼬리가 눌리며, T=1.5 는 반대. top-k 10 은 10번째 뒤가 정확히 0, top-p 0.9 는 몇 개가 남는지 범례의 숫자로 확인하라.

**해 보기**: `settings` 에 `"top-p 0.5": dict(top_p=0.5)` 를 추가하면 후보가 몇 개로 줄어드는지, `"T=0.3"` 을 넣으면 1등이 몇 % 가 되는지 보라.

- **temperature**: logits 를 T 로 나눈다. T<1 이면 1등에 몰리고(안전·단조), T>1 이면 평평해진다(다양·산만). T→0 은 argmax(greedy).
- **top-k**: 상위 k 개만 남기고 나머지 0. 꼬리를 통째로 자르지만 k 가 고정이라 "확실한 상황"에서도 k 개를 남긴다.
- **top-p (nucleus)**: 누적 확률이 p 가 될 때까지만 남긴다. 확실하면 1~2개, 애매하면 수십 개, 상황에 맞춰 k 가 변한다.
- **반복 억제**: 이미 나온 토큰의 점수를 깎는다. 작은 모델이 "…하고 하고 하고" 에 빠지는 것을 막는다.

## 3. 생성: 설정별 비교

In [ ]:
prompts = ["옛날 옛적에", "그는 아버지의 얼굴을 바라보며", "서울로 올라온 지 삼 년이 되던 해"]
configs = {
    "greedy (T=0)": dict(temperature=0.0),
    "T=1.0 그대로": dict(temperature=1.0),
    "T=0.8 · top-k 50": dict(temperature=0.8, top_k=50),
    "T=0.8 · top-p 0.9 · 반복억제 1.2": dict(temperature=0.8, top_p=0.9, repetition_penalty=1.2),
}
for name, kw in configs.items():
    print(f"===== {name}")
    for pr in prompts[:2]:
        g = torch.Generator().manual_seed(1337)
        out = generate_text(model, tok, pr, max_new_tokens=70, generator=g, **kw)
        print(out.replace("\n", " ⏎ "))
        print("---")

greedy 는 같은 구절을 맴돌기 쉽고, T=1 그대로는 꼬리에서 이상한 토큰이 섞인다. **T 0.8 + top-k 50** 또는 **top-p 0.9** 가 이 크기의 모델에서 가장 읽을 만하다.
1930년대 문체와 어휘(영채·형식·초봉이…)가 그대로 배어 나온다. 학습 데이터가 모델을 결정한다는 0장의 말이 여기서 확인된다.

## 4. 같은 프롬프트, 시드 다섯 번

In [ ]:
for seed in range(5):
    g = torch.Generator().manual_seed(seed)
    print(f"[{seed}]", generate_text(model, tok, "옛날 옛적에 호랑이가", max_new_tokens=40, temperature=0.8, top_k=50, generator=g).replace("\n", " "))

## 5. 데이터가 바뀌면: small-cpu vs mixed-cpu (ADR-0003)

같은 모델 크기, 같은 프롬프트, 같은 설정. 다른 것은 학습 데이터(근대문학 49만 토큰 vs 근대문학 + 위키 약 400만 토큰)와 토크나이저뿐이다.

In [ ]:
for run in ("small-cpu", "mixed-cpu"):
    try:
        m, ck = load_checkpoint(run, "best.pt")
    except FileNotFoundError:
        print(f"[{run}] 체크포인트 없음. scripts/train.py --config configs/{run}.yaml")
        continue
    rtok = BPETokenizer.load(TOKENIZER_DIR / ck.get("tokenizer", "bpe-8192.json"))
    print(f"===== {run} (step {ck['step']}, 토크나이저 {ck.get('tokenizer')})")
    for pr in ("옛날 옛적에 호랑이가", "서울은 대한민국의 수도로,", "그는 아버지의 얼굴을 바라보며"):
        g = torch.Generator().manual_seed(1337)
        print(generate_text(m, rtok, pr, max_new_tokens=60, temperature=0.8, top_k=50, generator=g).replace("\n", " ⏎ "))
        print("---")

근대문학 프롬프트에서는 두 모델 다 소설투를 잇지만, "서울은 대한민국의 수도로," 같은 현대 한국어·백과사전식 프롬프트는 mixed 쪽만 이어 쓴다.
데이터가 곧 모델의 세계다.

## 정리

- 생성 = `forward → 마지막 자리 logits → 조정 → softmax → multinomial → 붙이기` 의 반복. 문맥이 block_size 를 넘으면 앞을 잘라 쓴다.
- 조정은 전부 logits 단계: 반복 억제 → temperature → top-k → top-p.
- 소형 모델의 실전값: T 0.7~0.9, top-k 40~50 또는 top-p 0.9, 반복 억제 1.1~1.3.
- 이 모델은 base 모델이다. 질문에 답하지 않고 **이어 쓴다**. 대화가 되려면 10장의 SFT 가 필요하다.

---
**다음 장**: 9장, 평가. perplexity, 모델 크기 vs 손실, 어휘 크기 비교.